# Семинар 10: генераторы, итераторы, itertools

**План занятия:**
1. Нерасказанное про классы — множественное наследование и `super()`
2. От классов к итераторам — как Python обходит коллекции
3. Генераторы и `yield`
4. Итераторы: пишем свои
5. `itertools` — батарейки для ленивых вычислений
6. Задания

---
## Глава 0: нерасказанное про классы

### Множественное наследование и миксины

Python поддерживает наследование от нескольких классов сразу. Один из популярных паттернов — **миксин (mixin)**: небольшой класс, который добавляет одну конкретную возможность, но не предназначен для самостоятельного использования.

В примере ниже `EmailMixin` — это именно миксин: он добавляет поле `email`, но сам по себе не имеет смысла без `BasePerson`.

In [ ]:
from dataclasses import dataclass


@dataclass
class EmailMixin:
    email: str


@dataclass
class BasePerson:
    name: str
    age: int


@dataclass
class Person(EmailMixin, BasePerson):
    def __str__(self):
        return f"{self.name}, age {self.age}, email {self.email}"

In [ ]:
me = Person(name="Milena", age=22, email="mkamskaya@hse.ru")
print(me)

### MRO — порядок разрешения методов

При множественном наследовании Python должен понять, в каком порядке искать метод (например, `__init__`). Для этого используется алгоритм **C3 linearization**, а порядок поиска называется **MRO (Method Resolution Order)**.

Посмотреть MRO можно через атрибут `__mro__`. `super()` всегда вызывает следующий класс в этой цепочке — не обязательно родительский.

In [ ]:
class A:
    def __init__(self):
        print("started call init A")
        super().__init__()
        print("ended call init A")

class B:
    def __init__(self):
        print("started call init B")
        super().__init__()
        print("ended call init B")

class C(A, B):
    def __init__(self):
        print("started call init C")
        super().__init__()
        print("ended call init C")

In [ ]:
# Посмотрим на порядок вызовов
c = C()
print()
print("MRO:", [cls.__name__ for cls in C.__mro__])

Видим, что `super()` в классе `A` вызывает не `object`, а `B` — потому что по MRO `B` стоит следующим. Это кооперативное множественное наследование.

### Обращение к родителю через `super()`

In [ ]:
from dataclasses import dataclass

@dataclass
class BasePerson:
    name: str
    age: int

    def __str__(self):
        return f"{self.name}, age {self.age}"


@dataclass
class Person(BasePerson):
    def __str__(self):
        result = super().__str__()  # вызываем __str__ из BasePerson
        return "Person: " + result


me = Person(name="Milena", age=22)
print(me)

**Вопрос:** как будет работать `super()`, если наследование множественное?

**Вопрос**: какой тут будет порядок?

In [ ]:
class A:
    def method(self):
        print("Класс А")

class B(A):
    def method(self):
        super().method()
        print("Класс B")

class C(A):
    def method(self):
        super().method()
        print("Класс C")

class D(B, C):
    def method(self):
        super().method()
        print("Класс D")

obj = D()
obj.method()

---
## Глава 1: от классов к итераторам

### Как работает цикл `for`?

Мы уже умеем писать классы с кастомными методами. Но задумывались ли вы, как именно Python понимает, что по объекту можно делать `for`?

```python
for x in [1, 2, 3]:
    print(x)
```

Под капотом Python делает ровно три вещи:
1. Вызывает `iter(obj)` → получает **итератор**
2. Снова и снова вызывает `next(iterator)` → получает следующий элемент
3. Когда итератор поднимает `StopIteration` → цикл заканчивается

То есть `for x in collection` — это просто синтаксический сахар над двумя функциями: `iter()` и `next()`.

In [ ]:
collection = [1, 2, 3, 4, 5]

# Вот что делает for под капотом:
list_iter = iter(collection)   # шаг 1: получить итератор

print(next(list_iter))         # шаг 2: запрашивать по одному
print(next(list_iter))
print(next(list_iter))
print(next(list_iter))
print(next(list_iter))

In [ ]:
# Шаг 3: когда элементы кончились — StopIteration
print(next(list_iter))

### Протокол итератора

Чтобы объект стал итератором, его класс должен реализовать **два специальных метода**:

| Метод | Что делает |
|---|---|
| `__iter__(self)` | Возвращает сам объект-итератор (обычно `return self`) |
| `__next__(self)` | Возвращает следующий элемент или поднимает `StopIteration` |

Объект, у которого есть `__iter__` (но не обязательно `__next__`), называется **итерируемым (iterable)**. Примеры: `list`, `str`, `dict`, `set`.

Объект, у которого есть и `__iter__`, и `__next__`, называется **итератором**.

> **Итерируемый** — это как книга: из неё можно получить итератор.  
> **Итератор** — это как закладка: помнит, где ты остановился, и двигается вперёд.

In [ ]:
# enumerate — тоже итератор!
it = enumerate(collection)

while True:
    try:
        print(next(it))
    except StopIteration:
        break

In [ ]:
# Файловые объекты — тоже итераторы!
# f = open("input.txt")
# for line in f:          # next(f) возвращает следующую строку
#     print(line)
# f.close()

### Пишем свой итератор

Раз мы умеем писать классы, давайте напишем класс, реализующий протокол итератора. Сделаем итератор чисел Фибоначчи.

In [ ]:
class FibonacciIterator:
    def __init__(self, max_number):
        self.prev = 1
        self.cur = 1
        self.num = 0
        self.max_number = max_number

    def __next__(self):
        if self.num == self.max_number:
            raise StopIteration   # сигнал: элементы кончились

        result = self.prev
        self.prev, self.cur = self.cur, self.prev + self.cur
        self.num += 1
        return result

    def __iter__(self):
        return self  # итератор возвращает сам себя

In [ ]:
fib_iter = FibonacciIterator(10)

for x in fib_iter:
    print(x)

Обратите внимание: `fib_iter` — это обычный класс, но благодаря `__iter__` и `__next__` по нему можно делать `for`, `next()`, передавать его в `list()`, `sum()` и т.д.

Ещё один пример — итератор, возводящий число в квадрат снова и снова (бесконечный):

In [ ]:
class SquareIterator:
    def __init__(self, initial_number):
        self.number_to_square = initial_number

    def __next__(self):
        self.number_to_square = self.number_to_square ** 2
        return self.number_to_square

    def __iter__(self):
        return self


sq_iter = SquareIterator(2)

print(next(sq_iter))  # 4
print(next(sq_iter))  # 16
print(next(sq_iter))  # 256
print(next(sq_iter))  # 65536

### Реализуем свой `enumerate`

Теперь напишем аналог встроенного `enumerate` — это хороший пример того, как итератор может оборачивать другой итератор.

In [ ]:
from collections.abc import Iterable


class MyEnumerate:
    def __init__(self, iterable: Iterable, start: int = 0):
        self.iterable = iter(iterable)  # получаем итератор из переданного объекта
        self.start = start

    def __next__(self):
        return_value = (self.start, next(self.iterable))  # next() сам поднимет StopIteration
        self.start += 1
        return return_value

    def __iter__(self):
        return self


for x in MyEnumerate(["abc", "cde", "def"]):
    print(x)

---
## Глава 2: генераторы

### Зачем они нужны?

Писать класс с `__iter__` и `__next__` каждый раз — многословно. Python предлагает более лаконичный способ: **генераторные функции**.

Генераторная функция — обычная функция, но вместо `return` использует **`yield`**. При вызове она не выполняется сразу, а возвращает объект-генератор. Каждый раз, когда у него запрашивают `next()`, функция выполняется до следующего `yield`, возвращает значение и **приостанавливается** — сохранив всё своё состояние (переменные, позицию выполнения).

> Генераторы — это "ленивые" итераторы: они вычисляют значения по требованию, а не заранее.

In [ ]:
# Простой пример: свой range
def my_range(a, b):
    while a < b:
        yield a   # вернуть значение и приостановиться
        a += 1    # это выполнится при следующем next()

In [ ]:
rng = my_range(2, 10)

print(next(rng))  # 2
print(next(rng))  # 3
print(next(rng))  # 4

In [ ]:
# Генератор — это полноценный итератор, по нему можно делать for
for x in my_range(2, 6):
    print(x)

### Числа Фибоначчи через генератор

Сравните с `FibonacciIterator` выше — логика та же, но кода значительно меньше:

In [ ]:
def generate_fib(max_number):
    fib_1, fib_2 = 1, 1
    yield fib_1
    yield fib_2

    for _ in range(2, max_number):
        fib_1, fib_2 = fib_2, fib_1 + fib_2
        yield fib_2


fibs = generate_fib(10)
for x in fibs:
    print(x)

### Генераторы одноразовые!

Важно помнить: генератор можно пройти только один раз. Как закладка в книге — двигается только вперёд.

In [ ]:
fibs = generate_fib(10)

for x in fibs:
    print(x)

print("--- второй проход ---")
for x in fibs:  # уже пустой!
    print(x)

In [ ]:
next(fibs)  # StopIteration

### `yield from` — делегирование генератора

Если нужно вернуть все элементы другого итерируемого объекта, можно использовать `yield from` вместо цикла с `yield`. Это особенно удобно при рекурсии.

In [ ]:
def traverse_dict(d):
    """Обходит вложенный словарь и возвращает все листовые значения."""
    if not isinstance(d, dict):
        yield d
    else:
        for v in d.values():
            yield from traverse_dict(v)  # делегируем рекурсивному вызову
            # это эквивалентно:
            # for x in traverse_dict(v):
            #     yield x


d = {
    "one": {
        "two": {
            "three": "four",
            "five": "six",
        },
    },
    "seven": "eight",
}

for x in traverse_dict(d):
    print(x)

### Генераторное выражение

Аналог list comprehension, но ленивый — не создаёт список в памяти сразу.

In [ ]:
# List comprehension — создаёт весь список сразу
squares_list = [x ** 2 for x in range(10)]

# Генераторное выражение — вычисляет по одному
squares_gen = (x ** 2 for x in range(10))

print(type(squares_list))  # list
print(type(squares_gen))   # generator

for x in squares_gen:
    print(x)

### Когда использовать генераторы, а когда — итераторы-классы?

| | Генераторная функция | Класс-итератор |
|---|---|---|
| Код | Компактный | Многословный |
| Состояние | Хранится в стеке функции | Явно в атрибутах класса |
| Многоразовость | ❌ Одноразовый | ✅ Можно сделать многоразовым |
| Доп. методы | ❌ Нет | ✅ Можно добавить любые |

В большинстве случаев генераторы удобнее. Класс нужен, если хочется переиспользовать итератор или добавить дополнительное поведение.

---
## Глава 3: itertools

Модуль `itertools` — стандартная библиотека Python, набор готовых ленивых итераторов для типичных задач. Всё, что в нём есть, работает лениво: не создаёт промежуточных списков.

Чтобы посмотреть результат в интерактивном режиме, оборачиваем в `list()`, но в реальном коде это обычно не нужно.

In [ ]:
import itertools

### Цепочки и комбинации

In [ ]:
# chain — склейка нескольких итерируемых объектов
a = [1, 2, 3]
b = ["abc", "cde"]
c = [(1, 2), (3, 4)]

for x in itertools.chain(a, b, c):
    print(x)

In [ ]:
# chain.from_iterable — то же самое, но принимает один итерируемый объект из итерируемых
a = ["abc", "cde"]

for x in itertools.chain.from_iterable(a):
    print(x)

In [ ]:
# product — декартово произведение (аналог вложенных циклов)
a = ["A", "T", "G", "C"]
b = [1, 2, 3]

print(list(itertools.product(a, b)))

In [ ]:
# combinations — сочетания без повторений
print(list(itertools.combinations(a, 2)))

In [ ]:
# permutations — перестановки
print(list(itertools.permutations(["A", "B", "C"])))

### Фильтрация и срезы

In [ ]:
# compress — фильтр по маске (1 = взять, 0 = пропустить)
a = "abcdef"
print("".join(itertools.compress(a, [1, 0, 1, 1, 0, 0])))

In [ ]:
a = [-1, -2, -3, 6, 2, 1, -2, -1]

# dropwhile — пропускает элементы, пока условие истинно, потом возвращает всё
print(list(itertools.dropwhile(lambda x: x < 0, a)))

# takewhile — берёт элементы, пока условие истинно, потом останавливается
print(list(itertools.takewhile(lambda x: x < 0, a)))

In [ ]:
# filterfalse — противоположность filter: оставляет элементы, где условие ЛОЖНО
a = [1, -1, 6, 3, -2, 0, -6]
print(list(itertools.filterfalse(lambda x: x < 0, a)))

In [ ]:
# islice — ленивый срез (как a[start:stop:step], но без создания нового списка)
a = [1, 2, 3, 4, 5, 6, 7]

for x in itertools.islice(a, 1, 5, 2):
    print(x)

### Агрегация

In [ ]:
from functools import reduce

a = [1, 7, 3, 2, 5, 3]

# accumulate — как reduce, но возвращает все промежуточные значения
print(list(itertools.accumulate(a)))            # накопленная сумма
print(list(itertools.accumulate(a, max)))       # накопленный максимум
print(reduce(lambda x, y: x + y, a))           # итоговая сумма через reduce

In [ ]:
# pairwise — пары соседних элементов (новое в Python 3.10)
a = ["A", "T", "G", "C"]
print(list(itertools.pairwise(a)))

In [ ]:
# starmap — как map, но каждый элемент — это кортеж аргументов
print(list(itertools.starmap(pow, [(2, 5), (3, 2), (10, 3)])))

### `groupby` — группировка

`groupby` группирует **последовательные** одинаковые элементы. Важно: данные должны быть **отсортированы** по ключу группировки, иначе одинаковые элементы, стоящие не рядом, попадут в разные группы.

In [ ]:
a = [2, -2, 2, -2, 1, -1, -1, 2, -2, 2, -2, 0, 2, 1]

# Группируем по абсолютному значению
for elem, group in itertools.groupby(a, key=abs):
    print(elem, "|", *group)

In [ ]:
# Задача: превратить "ABBBCC" в "1A3B2C"
s = "ABBBCC"

result = "".join(
    f"{len(list(group))}{symbol}"
    for symbol, group in itertools.groupby(s)
)
print(result)

In [ ]:
# Группировка словаря по значениям
from operator import itemgetter

d = {
    "a": 1, "b": 2, "c": 3,
    "d": 1, "e": 2, "f": 3,
}

# Сначала сортируем, потом группируем!
for value, items in itertools.groupby(sorted(d.items(), key=itemgetter(1)), key=itemgetter(1)):
    print(value, ":", *map(itemgetter(0), items))

### Бесконечные итераторы

In [ ]:
# cycle — бесконечно повторяет коллекцию
a = ["A", "T", "G", "C"]
cyc = itertools.cycle(a)

for _ in range(7):
    print(next(cyc))

In [ ]:
# Пример применения cycle — консольный спиннер
import itertools
import sys
import time

def spinner(seconds):
    symbols = itertools.cycle('-\\|/')
    tend = time.time() + seconds
    while time.time() < tend:
        sys.stdout.write('\rPlease wait... ' + next(symbols))
        sys.stdout.flush()
        time.sleep(0.1)
    print()

spinner(2)

### Пример: выбрать тройку с наибольшим произведением

In [ ]:
import functools

a = [3, 1, 4, 1, 5, 9, 2, 6]
n = 3  # ищем тройку

best = max(
    itertools.combinations(a, n),
    key=functools.partial(functools.reduce, lambda x, y: x * y),
)
print(best)

### Краткая шпаргалка по `itertools`

| Функция | Что делает |
|---|---|
| `chain(*iters)` | Склейка нескольких итерируемых объектов |
| `chain.from_iterable(iter)` | То же, но из одного итерируемого объекта |
| `product(*iters)` | Декартово произведение |
| `combinations(iter, r)` | Сочетания по r |
| `permutations(iter)` | Все перестановки |
| `compress(iter, mask)` | Фильтр по маске из 0 и 1 |
| `dropwhile(pred, iter)` | Пропускать, пока условие истинно |
| `takewhile(pred, iter)` | Брать, пока условие истинно |
| `filterfalse(pred, iter)` | Обратный `filter` |
| `islice(iter, ...)` | Ленивый срез |
| `accumulate(iter)` | Накопленные значения (как `reduce` с историей) |
| `pairwise(iter)` | Пары соседних элементов |
| `starmap(func, iter)` | `map`, где аргументы упакованы в кортежи |
| `groupby(iter, key)` | Группировка соседних элементов |
| `cycle(iter)` | Бесконечный повтор |


---
## Задания

### Задание 1: многоразовый итератор

Написать итератор, по которому можно проходиться неограниченное количество раз. При этом допускается хранение элементов, но лишь столько, сколько мы максимально уже просмотрели.

```python
it = MyIterator([1, 2, 3])
for x in it: print(x)  # 1 2 3
for x in it: print(x)  # 1 2 3  <- снова работает!
```

> *Подсказка:* `__iter__` не обязан возвращать `self` — он может создавать новый объект.

In [ ]:
# Ваш код здесь

### Задание 2: генератор `chain`

Написать свой генератор `chain`, который принимает через `*` список коллекций и возвращает лениво их итеративную склейку:

```python
chain([1, 2, 3], {"a", "b", "c"}) -> 1, 2, 3, a, b, c
```

In [ ]:
from collections.abc import Iterable

def my_chain(*iterables: Iterable):
    ...


for x in my_chain([1, 2, 3], {"a", "b", "c"}):
    print(x)

### Задание 3: генератор `flatten`

Написать генератор `flatten`, принимающий коллекцию с вложенными итерируемыми объектами и возвращающий "сплющенный" поток элементов:

```python
[[1, 2, 3], [4, [5, 6]]] -> 1, 2, 3, 4, 5, 6
```

Проверять на итерируемость можно через `isinstance(x, Iterable)`.  
Не забудьте исключить строки — они тоже `Iterable`, но делить их на символы обычно не нужно.

In [ ]:
from collections.abc import Iterable

def flatten(iterable: Iterable):
    ...


print(list(flatten([[1, 2, 3], [4, [5, 6]]])))

### Задание 4 (со звёздочкой): `pairwise` через `itertools`

Используя `itertools.pairwise`, найти все числа из списка, которые **больше предыдущего** (то есть элементы, после которых идёт рост).

```python
a = [3, 1, 4, 1, 5, 9, 2, 6]
# Ожидаемый результат: [4, 5, 9, 6]
```

In [ ]:
from operator import itemgetter

a = [3, 1, 4, 1, 5, 9, 2, 6]

# Ваш код здесь

---
### Что почитать?

- **`more-itertools`** — ещё больше готовых инструментов: https://more-itertools.readthedocs.io/en/stable/
- **`pydash`** — функциональный стиль в Python, вдохновлённый lodash: https://pydash.readthedocs.io/en/latest/
- Статья про продвинутые паттерны с итераторами и декораторами: https://www.bbayles.com/index/decorator_factory